# OpenAI Chat Completions API Tutorial

This notebook demonstrates how to:
- Send requests to the OpenAI Chat Completions API
- Use dynamic prompts with f-strings
- Inspect API responses
- Calculate API costs
- Control completion length
- Use system, user, and assistant roles
- Maintain conversation history

## 1. Basic Chat Completion

First, initialize the OpenAI client and send a simple prompt.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

prompt = """
Your prompt goes here
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=100
)

print(response.choices[0].message.content)

## 2. Dynamic Prompts with f-Strings

You can insert variables directly into a prompt using Python f-strings.

In [ ]:
finance_text = """
Financial markets allow companies and governments to raise capital.
Investors can buy assets such as stocks and bonds in the expectation
of receiving returns.
"""

prompt = f"""Summarize the following text into two concise bullet points:
{finance_text}"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=100
)

print(response.choices[0].message.content)

## 3. Exploring the Response

The response object contains more than just the generated text. It also contains information such as token usage and the model used.

In [ ]:
print(response)

print("Model:", response.model)
print("Response ID:", response.id)
print("Generated text:", response.choices[0].message.content)
print("Usage:", response.usage)

## 4. Cost Calculation

API costs depend on the number of input and output tokens. The `usage` object provides the actual token counts used by the request.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=100
)

# Example prices in USD per token
input_token_price = 0.15 / 1_000_000
output_token_price = 0.60 / 1_000_000

# Extract actual token usage
input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens

# Calculate estimated cost
cost = (
    input_tokens * input_token_price
    + output_tokens * output_token_price
)

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Estimated cost: ${cost:.8f}")

## 5. Limiting Output Tokens

Setting a maximum completion length can help control response size and API costs.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=100,
    temperature=1
)

print(response.choices[0].message.content)

## 6. Roles in Chatbots

Chat-based applications use different message roles:

- **system** — defines the assistant's behavior and instructions
- **user** — represents the user's request
- **assistant** — represents previous assistant responses

A system message can be used to control how the model behaves.

In [ ]:
sys_msg = """You are a finance education assistant that helps students study for exams.
If you are asked for specific, real-world financial advice with risk to their finances,
respond with:
"I'm sorry, I am not allowed to provide financial advice."""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": "Which stocks should I invest in?"}
    ]
)

print(response.choices[0].message.content)

## 7. Using Assistant Messages

An assistant message can be included to demonstrate previous conversation context or establish an example response.

The conversation should always end with a user message when requesting a new assistant response.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a Python programming tutor who speaks concisely."
        },
        {
            "role": "user",
            "content": "How do you define a Python list?"
        },
        {
            "role": "assistant",
            "content": "Lists are defined by enclosing a comma-separated sequence of objects inside square brackets [ ]."
        },
        {
            "role": "user",
            "content": "What is the difference between mutable and immutable objects?"
        }
    ]
)

print(response.choices[0].message.content)

## 8. Maintaining Conversation History

For a chatbot, we can store previous user and assistant messages in a list and send the complete conversation history with each request.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a data science tutor who provides short, simple explanations."
    }
]

user_qs = [
    "Why is Python so popular?",
    "Summarize this in one sentence."
]

for q in user_qs:
    print("User:", q)

    user_dict = {
        "role": "user",
        "content": q
    }

    messages.append(user_dict)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    assistant_dict = {
        "role": "assistant",
        "content": response.choices[0].message.content
    }

    messages.append(assistant_dict)

    print("Assistant:", response.choices[0].message.content)
    print()

## Key Takeaways

1. Prompts are sent through the `messages` parameter.
2. System messages define assistant behavior.
3. User messages contain requests.
4. Assistant messages preserve conversation context.
5. `response.usage` provides actual token consumption.
6. Limiting completion tokens can reduce output size and cost.
7. Conversation history can be maintained by appending messages to a list.